In [1]:
import pandas as pd
import numpy as np


In [6]:
df = pd.read_excel('DA_BH.xlsx', sheet_name = 'Inventory_Store')
df_WH = pd.read_excel('DA_BH.xlsx', sheet_name = 'Inventory_WH')

df_sale = pd.read_csv('sales_clean.csv')

In [7]:
df_sale['date'] = pd.to_datetime(df_sale['date'])

In [8]:
#lấy 30 ngày gần nhất 
sales_30d = df_sale[
    (df_sale['date'] >= '2026-09-01') &
    (df_sale['date'] <= '2026-09-30') &
    (df_sale['qty'] > 0)
].copy()

In [10]:
sales_30d_sku_store = (
    sales_30d
    .groupby(['store_id', 'sku'])['qty']
    .sum()
    .reset_index(name = 'units_30d')
)

In [11]:
sales_30d_sku_store.head()

,store_id,sku,units_30d
0,CT01,SKU001,6
1,CT01,SKU002,1
2,CT01,SKU003,1
3,CT01,SKU004,1
4,CT01,SKU009,4


In [12]:
sales_30d_sku_store['avg_daily_sale'] = (
    sales_30d_sku_store['units_30d'] / 30
    
)
sales_30d_sku_store.head()

,store_id,sku,units_30d,avg_daily_sale
0,CT01,SKU001,6,0.200000
1,CT01,SKU002,1,0.033333
2,CT01,SKU003,1,0.033333
3,CT01,SKU004,1,0.033333
4,CT01,SKU009,4,0.133333


In [15]:
inventory_store = pd.read_excel(
    'DA_BH.xlsx',
    sheet_name = 'Inventory_Store'
)
inv = inventory_store.merge(
    sales_30d_sku_store,
    on = ['store_id', 'sku'],
    how = 'left'
)
inv.head()

,store_id,sku,on_hand,in_transit,units_30d,avg_daily_sale
0,HN01,SKU003,6,6,5.0,0.166667
1,HN01,SKU004,1,0,5.0,0.166667
2,HN01,SKU005,6,6,4.0,0.133333
3,HN01,SKU006,60,0,1.0,0.033333
4,HN01,SKU007,16,6,3.0,0.100000


In [17]:
inv['DOH'] = np.where(
    inv['avg_daily_sale'] > 0,
    inv['on_hand'] / inv['avg_daily_sale'],
    np.inf
)

In [18]:
inv.head()

,store_id,sku,on_hand,in_transit,units_30d,avg_daily_sale,DOH
0,HN01,SKU003,6,6,5.0,0.166667,36.0
1,HN01,SKU004,1,0,5.0,0.166667,6.0
2,HN01,SKU005,6,6,4.0,0.133333,45.0
3,HN01,SKU006,60,0,1.0,0.033333,1800.0
4,HN01,SKU007,16,6,3.0,0.100000,160.0


In [ ]:
#phân loại tồn kho
inv['Phân loại tồn kho'] = np.select(
    [
        inv['DOH'] < 14,
        inv['DOH'] > 90
    ],
    [
        'Sắp hết',
        'Chậm bán'
    ],
    default = 'Bình thường'
    
)
inv['projected_stock'] = inv['on_hand'] + inv['in_transit']

inv['projected_DOH'] = np.where(
    inv['avg_daily_sale'] > 0,
    inv['projected_stock'] / inv['avg_daily_sale'],
    np.inf
)
inv.head()